In [1]:
###################################
# IMPORT
###################################

from polars import col, concat, lit, when
import polars as pl

import sys; sys.path.append('.'); 
import scripts.hdb_helpers as dc

In [2]:
###################################
# DATA LOAD
###################################
hdbdata_transformed = pl.read_parquet('datalake/hdb/transformed/hdbdata')
hdbdata_transformed.glimpse()

Rows: 77254
Columns: 14
$ month               <str> '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01'
$ town                <str> 'BUKIT MERAH', 'QUEENSTOWN', 'KALLANG/WHAMPOA', 'BUKIT MERAH', 'PASIR RIS', 'BUKIT MERAH', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'MARINE PARADE', 'CENTRAL AREA'
$ flat_type           <str> '5 ROOM', 'EXECUTIVE', 'EXECUTIVE', '5 ROOM', 'EXECUTIVE', '5 ROOM', '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '45', '150', '46', '23', '130', '2C', '57', '73', '72', '533'
$ street_name         <str> 'LENGKOK BAHRU', 'MEI LING ST', 'BENDEMEER RD', 'JLN MEMBINA', 'PASIR RIS ST 11', 'BOON TIONG RD', "JLN MA'MOR", 'MARINE DR', 'MARINE DR', 'UPP CROSS ST'
$ storey_range        <str> '01 TO 03', '07 TO 09', '16 TO 18', '13 TO 15', '07 TO 09', '13 TO 15', '01 TO 03', '22 TO 24', '22 TO 24', '10 TO 12'
$ floor_area_sqm      <f64> 139.0, 147.0, 146.0, 110.0, 188.0, 115.0, 128.0, 120.0, 117.0, 120.0


In [3]:
'''
REQUIREMENT: Hash this identifier column using an irreversible hashing algorithm, while preserving its
uniqueness. Explain the hashing algorithm that you adopted in your documentations.
'''

'''
We use SHA256. It is ubiquiously known to be the go to. 
SHA-256 mangles any input through 64 rounds of math into a fixed, one-way 256-bit fingerprint — you can't reverse it or fake a collision.
It's the industry default today running two decades zero known breaks.
Mathematically: The function is one-way because it's many-to-one — many possible inputs can map to the same fixed-size output, 
so there's no *unique* input to reverse back to. For any one string, you will get a humongous number of random inputs. larger than the counts of sand on earth.
'''

"\nWe use SHA256. It is ubiquiously known to be the go to. \nSHA-256 mangles any input through 64 rounds of math into a fixed, one-way 256-bit fingerprint — you can't reverse it or fake a collision.\nIt's the industry default today running two decades zero known breaks.\nMathematically: The function is one-way because it's many-to-one — many possible inputs can map to the same fixed-size output, \nso there's no *unique* input to reverse back to. For any one string, you will get a humongous number of random inputs. larger than the counts of sand on earth.\n"

In [4]:
import hashlib

In [5]:
# # ════════════════════ SCRATCH  ══════════════════════════════════════════════════════════════════════════════════════════
# Author's note: we could do SHA256 in in polars function itself, but I would prefer to make the call to not do so with reason:
# 1) we dont want only people who know language well to be able to understand the process without looking up the terms
# 2) even if they could, there are too many nests and a polars expert would take more than seconds to confirm what is happening
# 3) list comprehension is a universal python method, much more audience can read it without looking up the terms, 
# and it has less nesting - speeding up understanding
# 4) there is no computational speed difference, nor is there even less typing comparing polars method to list comprehension
####

# hdbdata_id_hashed = (hdbdata_transformed
#                             .with_columns(
#                                 identifier_hash = col('resale_identifier')
#                                 .map_batches(
#                                     lambda s: pl.Series([hashlib.sha256(x.encode()).hexdigest() for x in s]), return_dtype=pl.String

#     ))
# )

# to_hash_list = hdbdata_transformed['resale_identifier'].to_list()
# hashed_list = [hashlib.sha256(x.encode()).hexdigest() for x in to_hash_list]

# hdbdata_id_hashed = (hdbdata_transformed
#                             .with_columns(identifier_hash = pl.Series(hashed_list))
#                             )
# # ════════════════════════════════════════════════════════════════════════════════════════════════════════

In [6]:
############################
#### Compute Hash Lookup ###
############################
to_hash_list = hdbdata_transformed['resale_identifier'].unique().to_list()
# validate
print(to_hash_list[1:10])

['S0643509K', 'S5565502J', 'S3516009C', 'S2113110J', 'S9403910J', 'S3725110H', 'S0258108B', 'S9426710T', 'S2213309B']


In [7]:
# Actual SHA256 HASH computation here
hash_lookup = [(x, hashlib.sha256(x.encode()).hexdigest()) for x in to_hash_list]
# validate
print(hash_lookup[1:10])

[('S0643509K', 'b0b9f8a104ad158e1a895e62255d5462d837d587138b87a9c2d69c97b95ad74e'), ('S5565502J', '3f4d9304275fe5af5ef3d4f07ff07a4411c55d3119969c69887d33d47cbd6b11'), ('S3516009C', '4fa83c72997c7fcbdb60334181d7dc613394e556e9680f7e7bc1b59269a7c807'), ('S2113110J', 'fd1e1d9471abc978e43dacdfc24818d6a3383564a5c371a22f678fbeafd0005d'), ('S9403910J', 'c770949c26f600f0a0d6afb5f4f76180502fc575210074494bd95e1c42051bab'), ('S3725110H', 'd90180311fd4438c0a7c25507f09bfcb2d2709c70bab9d9b9f98580037f33daf'), ('S0258108B', 'e5e23ef6722cda2fa7f50810c6fe25958854e7a4ba7faf6d141189c235470dab'), ('S9426710T', '5e01fde13d15beb5c208962b6231e22628455c61d04fca17676122a97704e0da'), ('S2213309B', '6831f5442cf48c30b58cbf18f3b5fd5190028f1bfe66722df87a1fdabb95a842')]


In [8]:
df_hash_lookup = (
    pl.DataFrame(hash_lookup, schema=['resale_identifier', 'identifier_hash'], orient='row')
)
# validate
print(df_hash_lookup)

shape: (77_254, 2)
┌───────────────────┬─────────────────────────────────┐
│ resale_identifier ┆ identifier_hash                 │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S7643608Y         ┆ bc8e53571bd76cf9a00d014b44db1a… │
│ S0643509K         ┆ b0b9f8a104ad158e1a895e62255d54… │
│ S5565502J         ┆ 3f4d9304275fe5af5ef3d4f07ff07a… │
│ S3516009C         ┆ 4fa83c72997c7fcbdb60334181d7dc… │
│ S2113110J         ┆ fd1e1d9471abc978e43dacdfc24818… │
│ …                 ┆ …                               │
│ S0223903B         ┆ d293286ce5130724dc627139ad93fc… │
│ S1536709T         ┆ 69fc041a9107e4d9c8f0547571d012… │
│ S8514905Y         ┆ d20f3212078babb9cadb1567121c0c… │
│ S1375804B         ┆ 5e7c188c6098c87bf8853de4905e04… │
│ S1896006S         ┆ 293059a28319e7615a75d8048cf486… │
└───────────────────┴─────────────────────────────────┘


In [9]:
############################
#### Join hash lookup to data ###
############################

In [10]:
hdbdata_transformed_plus_hash = hdbdata_transformed.join(df_hash_lookup, on='resale_identifier', how='left')

dc.unicity(hdbdata_transformed_plus_hash, 'identifier_hash')

victims will be returned in global as uvictims
good!: multiplicity passed!:
unicity ended


In [11]:
# validate: row count should be unchanged by the join, and hash values should match the map_batches version
print(f"row count before join: {hdbdata_transformed.height:,}")
print(f"row count after join:  {hdbdata_transformed_plus_hash.height:,}")

row count before join: 77,254
row count after join:  77,254


In [12]:
# validate
hdbdata_transformed_plus_hash.select('record_id', 'resale_identifier', 'identifier_hash').show(5)

record_id,resale_identifier,identifier_hash
str,str,str
"""002-366781""","""S0457301B""","""5fba7fd7a6712ff7d75b04571bf45c…"
"""002-367459""","""S1508601Q""","""3525c8b4ae901eaa138e9fb96f2e15…"
"""002-367318""","""S0467901K""","""3e39d7b88bec804d531f3a0289099e…"
"""002-366779""","""S0237301B""","""718d6f8640272b4a521f77962eab18…"
"""002-367381""","""S1306601P""","""efa5c231993c21e3112fdbb86d23fb…"


In [13]:
print(f"n unique identifier: {hdbdata_transformed_plus_hash['resale_identifier'].n_unique():,}")
print(f"n unique identifier_hash: {hdbdata_transformed_plus_hash['identifier_hash'].n_unique():,}")
print(f"total rows: {hdbdata_transformed_plus_hash.height:,}")

# validate look of data
dc.showall(hdbdata_transformed_plus_hash, 'closey')

n unique identifier: 77,254
n unique identifier_hash: 77,254
total rows: 77,254
shape: (77_254, 15)
┌─────────┬─────────────────┬───────────┬───────┬─────────────────┬──────────────┬────────────────┬────────────────────┬─────────────────────┬──────────────┬─────────────────┬────────────────────┬────────────┬───────────────────┬────────────────────┐
│ month   ┆ town            ┆ flat_type ┆ block ┆ street_name     ┆ storey_range ┆ floor_area_sqm ┆ flat_model         ┆ lease_commence_year ┆ resale_price ┆ tabling_version ┆ remaining_lease    ┆ record_id  ┆ resale_identifier ┆ identifier_hash    │
│ ---     ┆ ---             ┆ ---       ┆ ---   ┆ ---             ┆ ---          ┆ ---            ┆ ---                ┆ ---                 ┆ ---          ┆ ---             ┆ ---                ┆ ---        ┆ ---               ┆ ---                │
│ str     ┆ str             ┆ str       ┆ str   ┆ str             ┆ str          ┆ f64            ┆ str                ┆ i64                 ┆ f64 

In [14]:
############################
#### Decisioning dataset ID columns to include in final data asset ###
############################
# We are here assuming that if the exploited dataset is the HASHED dataset, then we do not want the end-users to see what the unhashed value is.
# So, we make the call to replace the column [resale_dentifier] as hashed completely

# we also decide to keep the [record_id], so that it can be traced back to source data in much quicker time, should users have raise bug complaints
hdbdata_cleanly = hdbdata_transformed_plus_hash.drop('resale_identifier').rename({'identifier_hash':'resale_identifier'})

In [15]:
#################
### DATA WRITE to Datalake
#################
OUTPUT = hdbdata_cleanly
OUTPUT.glimpse()
OUTPUT.write_parquet('datalake/hdb/hashed/hdbdata', partition_by=['tabling_version', 'month'], mkdir=True)
print('written parquet')




Rows: 77254
Columns: 14
$ month               <str> '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01'
$ town                <str> 'BUKIT MERAH', 'QUEENSTOWN', 'KALLANG/WHAMPOA', 'BUKIT MERAH', 'PASIR RIS', 'BUKIT MERAH', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'MARINE PARADE', 'CENTRAL AREA'
$ flat_type           <str> '5 ROOM', 'EXECUTIVE', 'EXECUTIVE', '5 ROOM', 'EXECUTIVE', '5 ROOM', '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '45', '150', '46', '23', '130', '2C', '57', '73', '72', '533'
$ street_name         <str> 'LENGKOK BAHRU', 'MEI LING ST', 'BENDEMEER RD', 'JLN MEMBINA', 'PASIR RIS ST 11', 'BOON TIONG RD', "JLN MA'MOR", 'MARINE DR', 'MARINE DR', 'UPP CROSS ST'
$ storey_range        <str> '01 TO 03', '07 TO 09', '16 TO 18', '13 TO 15', '07 TO 09', '13 TO 15', '01 TO 03', '22 TO 24', '22 TO 24', '10 TO 12'
$ floor_area_sqm      <f64> 139.0, 147.0, 146.0, 110.0, 188.0, 115.0, 128.0, 120.0, 117.0, 120.0


written parquet
